<a href="https://colab.research.google.com/github/mjh5153/pgp-generative-ai/blob/main/LLama_Course_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# LLamindex
# embeddings-type, vector-stores
# Failures - windows - package not available ? install from command line
!pip install python-dotenv llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-vector-stores-faiss faiss-cpu llama-index-readers-file nest-asyncio

Custom Vector Stores - developers.llamaindex.ai/llamaparse/cloud-index-v2/

In [7]:
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
from llama_index.core import (
    # read file from directory
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    PromptTemplate
) # parent level

#3. LLM level

from llama_index.core.node_parser import SentenceSplitter # llama doesn't have textsplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.faiss import FaissVectorStore # vector store

import faiss
import os # operating system

# load api keys
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

if not GOOGLE_API_KEY:
    raise ValueError("No API key was found. Please either create a .env file with GOOGLE_API_KEY=your_key or set it in Google Colab Secrets (the key icon on the left panel) with the name GOOGLE_API_KEY.")

# load the document ==> text document (data.txt)
documents = SimpleDirectoryReader(input_files=["data.txt"], encoding="utf-8").load_data()

# Text splitter
text_splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50) # GPU ideal, power consumption

# embedding
# Embeddings ==> using google's gemini-embedding-001 - Mofication model - GoogleGenAIEmbedding
embeddings = GoogleGenAIEmbedding(model_name="models/gemini-embedding-001", api_key=GOOGLE_API_KEY)

# Vector store - small mod from LangGraph
dim = len(embeddings.get_text_embedding("hello"))
vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(dim))
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(documents,
    storage_context=storage_context,
    embed_model=embeddings,
    transformations=[text_splitter])

# llm model
llm = GoogleGenAI(model="models/gemini-3.8-flash", api_key=GOOGLE_API_KEY)

# prompt
prompt = PromptTemplate(
    "Answer the question based only on the context:\n\n{context_str}\n\nQuestion: {query_str}"
)

# rag_pipline
query_engine = index.as_query_engine(
    llm=llm,
    text_qa_template=prompt,
    similarity_top_k=4,
)

query = "What is this document all about?"

response = await query_engine.aquery(query)
print(response)

Based on the context, this document is about Lewis Carroll's 1865 children's novel ***Alice's Adventures in Wonderland***. 

Specifically, it details:
* An overview of the book, its plot, literary nonsense genre, and original illustrations by John Tenniel.
* Its positive reception, cultural impact, lasting legacy, translations, adaptations, and sequels.
* The origins and conception of the story during an 1862 river trip with the Liddell children, Carroll's relationship with the children (especially Alice Liddell), and how the story was originally created and written down for her.


In [19]:
dir(index.as_query_engine)

NameError: name 'index' is not defined